In [68]:
#!python -m venv .venv
#!source .venv/bin/activate
#!pip install pygame-ce ipykernel

import os, pickle, socket, struct, threading, time
import pygame
import random
import shelve


from netcode import NetClient

pygame.init()


(5, 0)

In [69]:
info = pygame.display.Info()
screen_width = info.current_w
screen_height = info.current_h
screen = pygame.display.set_mode((screen_width, screen_height))

#Screen sized fixed
backgroundImage = pygame.image.load('redSky.jpg').convert()
backgroundImage = pygame.transform.scale(backgroundImage, (screen_width, screen_height))
backgroundImage2 = pygame.image.load('blueSky.jpg').convert()
backgroundImage2 = pygame.transform.scale(backgroundImage2, (screen_width, screen_height))


winImage = pygame.image.load('you-win.jpg').convert()
winImage = pygame.transform.scale(winImage, (screen_width//2, screen_height//2))
winImage.set_alpha(245)

gameOverImage = pygame.image.load('GameOver.png').convert_alpha()
gameOverImage = pygame.transform.scale(gameOverImage, (screen_width//2, screen_height//2))
gameOverImage.set_alpha(245)




terrain  = pygame.image.load(("terrainSheet.png")).convert_alpha()
platform = terrain.subsurface(pygame.Rect(272, 16, 48, 16))
platform = pygame.transform.scale_by(platform, 3)


world_tileset = pygame.image.load("world_tileset.png").convert_alpha()
ground_tile = world_tileset.subsurface(pygame.Rect(0, 0, 16, 16))
ground_tile = pygame.transform.scale_by(ground_tile, 3)
below_tile = world_tileset.subsurface(pygame.Rect(0, 16, 16, 16))
below_tile = pygame.transform.scale_by(below_tile, 3)


pumpkin = world_tileset.subsurface(pygame.Rect(64, 128, 16, 16))
pumpkin = pygame.transform.scale_by(pumpkin, 3)
bush = world_tileset.subsurface(pygame.Rect(16, 48, 16, 16))
bush = pygame.transform.scale_by(bush, 3)
exclamation_mark = world_tileset.subsurface(pygame.Rect(16, 32, 16, 16))
exclamation_mark = pygame.transform.scale_by(exclamation_mark, 3)
orange_bush = world_tileset.subsurface(pygame.Rect(80, 112, 16, 16))
orange_bush = pygame.transform.scale_by(orange_bush, 3)


characters = pygame.image.load("CharSprites.png").convert_alpha()
enemyImage = pygame.image.load("YellowBird-Idle.png").convert_alpha()
enemyImage = enemyImage.subsurface(pygame.Rect(64,0,32,32))
enemyImage =  pygame.transform.scale_by(enemyImage, 2.5)





In [70]:
char_list =  [(kolumn, rad) for rad in range(4) for kolumn in range(4)]
def pick_character(choice: int):
    x, y = char_list[choice-1]
    player_character = characters.subsurface(pygame.Rect(x*64, y*64, 64, 64))
    player_character =  pygame.transform.scale_by(player_character, 1.5)
    return player_character

In [71]:
PLATFORM_WIDTH, PLATFORM_HEIGHT = platform.get_size()

PLATFORMS = [
    (480, 872),
    (1216, -1160),
    (984, -1656),
    (484, -3356),
    (352, -4592),
    (1060, -4952),
    (484, -5848),
    (840, -6032),
    (1184, -6208),
    (256, -6932),
    (896, -7308),
    (856, -8068),
    (200, -8452),
    (488, -8644),
    (1184, -9016),
    (972, -9804), 
]


In [72]:
GROUND_WIDTH, GROUND_HEIGHT = ground_tile.get_size()

GROUND = [
    (x, 1080 - GROUND_HEIGHT) for x in range(-100, 3000, GROUND_WIDTH)
]

In [73]:
BELOW_WIDTH, BELOW_HEIGHT = below_tile.get_size()
BELOW_ROWS = 10

BELOW = [
    (x, 1080 - BELOW_HEIGHT + 48 + row * BELOW_HEIGHT)
    for row in range(BELOW_ROWS)
    for x in range(0, 3000, BELOW_WIDTH)
]

In [74]:
def make_island(x, y, width, height):
    assert 2 <= width <= 18, "island width should be 2-18 blocks"
    assert 2 <= height <= 10, "island height should be 2-10 blocks"

    ground_tiles = [(x + col * GROUND_WIDTH, y) for col in range(width)]

    below_tiles = [
        (x + col * BELOW_WIDTH, y + row * BELOW_HEIGHT)
        for row in range(1, height)
        for col in range(width)
    ]

    return ground_tiles, below_tiles

In [75]:
ISLANDS = [
    (720, 724, 13, 3),
    (1452, 564, 9, 3),
    (872, 416, 7, 3),
    (420, 256, 8, 3),
    (1084, 104, 4, 3),
    (572, -52, 5, 3),
    (1176, -208, 15, 3),
    (6, -368, 15, 3),
    (1484, -458, 5, 2),
    (776, -684, 13, 3),
    (900, -882, 8, 2),
    (1564, -992, 6, 3),
    (580, -1320, 11, 3),
    (1296, -1488, 11, 3),
    (468, -1820, 9, 3),
    (1060, -1992, 8, 3),
    (500, -2152, 8, 3),
    (76, -2324, 6, 3),
    (524, -2492, 11, 3),
    (1236, -2660, 11, 3),
    (560, -2824, 10, 3),
    (1236, -3004, 9, 3),
    (812, -3184, 5, 3),
    (44, -3536, 6, 3),
    (472, -3716, 6, 3),
    (856, -3884, 8, 3),
    (1424, -4064, 6, 3),
    (936, -4240, 6, 3),
    (600, -4416, 5, 3),
    (704, -4772, 5, 3),
    (1352, -5128, 7, 3),
    (956, -5304, 4, 3),
    (508, -5492, 7, 3),
    (84, -5672, 5, 3),
    (1456, -6388, 4, 3),
    (956, -6564, 7, 3),
    (552, -6748, 5, 3),
    (568, -7116, 4, 3),
    (1216, -7496, 3, 3),
    (1584, -7688, 5, 3),
    (1164, -7872, 4, 3),
    (484, -8260, 3, 3),
    (764, -8828, 4, 3),
    (1552, -9208, 3, 3),
    (1156, -9408, 4, 3),
    (656, -9604, 5, 3),
    (1200, -10000, 15, 5),
]

for ix, iy, iw, ih in ISLANDS:
    island_ground, island_below = make_island(ix, iy, iw, ih)
    GROUND += island_ground
    BELOW += island_below

In [76]:
PUMPKIN_WIDTH, PUMPKIN_HEIGHT = pumpkin.get_size()

def make_pumpkins(islands, seed=1334):
    rng = random.Random(seed)
    ground_y = 1080 - GROUND_HEIGHT - PUMPKIN_HEIGHT
    pumpkins = []

    x = 500
    pumpkins.append((x, ground_y))
        

    island_index = rng.randint(3, 6)
    while island_index < len(islands):
        ix, iy, iwidth, _ = islands[island_index]
        px = ix + rng.randint(0, iwidth - 1) * GROUND_WIDTH
        py = iy - PUMPKIN_HEIGHT
        pumpkins.append((px, py))
        island_index += rng.randint(3, 6)

    return pumpkins

PUMPKINS = make_pumpkins(ISLANDS)

In [77]:
BUSH_WIDTH, BUSH_HEIGHT = bush.get_size()

def make_bushes(islands, seed=1334):
    rng = random.Random(seed)
    ground_y = 1080 - GROUND_HEIGHT - BUSH_HEIGHT
    bushes = []

    x = 1000
    bushes.append((x, ground_y))
        

    island_index = rng.randint(1, 2)
    while island_index < len(islands):
        ix, iy, iwidth, _ = islands[island_index]
        px = ix + rng.randint(0, iwidth - 1) * GROUND_WIDTH
        py = iy - BUSH_HEIGHT
        bushes.append((px, py))
        island_index += rng.randint(1, 2)

    return bushes

BUSHES = make_bushes(ISLANDS)

In [78]:
ORANGE_BUSH_WIDTH, ORANGE_BUSH_HEIGHT = orange_bush.get_size()

def make_bushes(islands, seed=1334):
    rng = random.Random(seed)
    ground_y = 1080 - GROUND_HEIGHT - ORANGE_BUSH_HEIGHT
    orange_bushes = []

    x = 50
    orange_bushes.append((x, ground_y))
        

    island_index = rng.randint(2, 5)
    while island_index < len(islands):
        ix, iy, iwidth, _ = islands[island_index]
        px = ix + rng.randint(0, iwidth - 1) * GROUND_WIDTH
        py = iy - BUSH_HEIGHT
        orange_bushes.append((px, py))
        island_index += rng.randint(2, 5)

    return orange_bushes

ORANGE_BUSHES = make_bushes(ISLANDS)

In [79]:
class Camera:

    def __init__(self):
        self.offset_x = 0
        self.offset_y = 0
        self.move_speed = 5
        self.image = backgroundImage

    def follow(self, player):
        #self.offset_x = screen.get_width() // 2 - player.x
        self.offset_y = screen.get_height() // 2 - player.y

    def death_screen(self):
        screen.blit(gameOverImage, (screen.get_width() // 4,screen.get_height() // 4 ))

    def win_screen(self, player):
        screen.blit(winImage, (screen.get_width() // 4,screen.get_height() // 4 ))
        player.x = 500
        player.y = 500
        pygame.display.flip()
        pygame.time.wait(3000)

    def render_world(self, platforms, ground, below, pumpkins,bushes, orange_bushes, player, enemies,checkPoint, remotes=()):
        self.follow(player)

        screen.fill((0, 0, 0))
        min_render_x = -100
        max_render_x = screen.get_width() +100
        min_render_y = player.y - screen.get_height() // 2 -100
        max_render_y = player.y + screen.get_height() // 2 +100
        
        if (player.y <= -5000):
            self.image = backgroundImage2
        elif(player.y > -5000):
            self.image = backgroundImage


        
        screen.blit(self.image, (0, 0))

        screen.blit(checkPoint.image, (checkPoint.x+self.offset_x, checkPoint.y+self.offset_y))

        for x, y in platforms:
            if min_render_x <= x and x <= max_render_x:
                if min_render_y <= y and y <= max_render_y:
                    screen.blit(platform, (x+self.offset_x, y+self.offset_y))

        for x, y in ground:
            if min_render_x <= x and x <= max_render_x:
                if min_render_y <= y and y <= max_render_y:
                    screen.blit(ground_tile, (x+self.offset_x, y+self.offset_y))
        
        for x, y in below:
            if min_render_x <= x and x <= max_render_x:
                if min_render_y <= y and y <= max_render_y:
                    screen.blit(below_tile, (x+self.offset_x, y+self.offset_y))

        for x, y in pumpkins:
            if min_render_x <= x and x <= max_render_x:
                if min_render_y <= y and y <= max_render_y:
                    screen.blit(pumpkin, (x+self.offset_x, y+self.offset_y))

        for x, y in bushes:
            if min_render_x <= x and x <= max_render_x:
                if min_render_y <= y and y <= max_render_y:
                    screen.blit(bush, (x+self.offset_x, y+self.offset_y))

        for x, y in orange_bushes:
            if min_render_x <= x and x <= max_render_x:
                if min_render_y <= y and y <= max_render_y:
                    screen.blit(orange_bush, (x+self.offset_x, y+self.offset_y))

        for e in enemies:
            if min_render_x <= e.x and e.x <= max_render_x:
                if min_render_y <= e.y and e.y <= max_render_y:
                    screen.blit(e.image, (e.x+ self.offset_x, e.y + self.offset_y))
       
        for p in remotes:
            screen.blit(pick_character(p.char), (p.x + self.offset_x, p.y + self.offset_y))
        
        screen.blit(player.image, (player.x + self.offset_x, player.y + self.offset_y))
        
        if(checkPoint.win(player)):
            self.win_screen(player)
                

In [80]:
class Player:
  def __init__(self, image, camera: Camera):
      #Koordinaterna
      self.x = 500
      self.y = 500
      self.speed = 10
      self.vel_y = 0
      self.gravity = 0.75
      self.jump_power = -20
      self.on_ground = False
      self.image = image
      self.camera = camera
      self.rect = pygame.Rect(self.x, self.y, image.get_width(), image.get_height())
      self.dead = False
      self.flipped = True
      self.timeToFlip = False

  def get_rect(self):
      return pygame.Rect(self.x, self.y, self.image.get_width(), self.image.get_height())

  def move(self, left, right, jump):
    #Writen here due to a bug that has appeared that might have to do with how pygame is built
    self.speed = 10
    
    if left:
      self.x -= self.speed
      if self.timeToFlip:
        self.flipped = True
        self.image = pygame.transform.flip(self.image, self.flipped, False)
        self.timeToFlip = False

     

    if right:
      self.x += self.speed
      self.image = pygame.transform.flip(self.image, self.flipped, False)
      if self.timeToFlip is False:
         self.timeToFlip = True
      self.flipped = False

    if jump and self.on_ground:
        self.vel_y = self.jump_power
        self.on_ground = False

  def apply_gravity(self):
      self.vel_y += self.gravity
      if self.vel_y >= 30:
            self.vel_y = 30
          

      self.y += self.vel_y
      if (self.y > 3000):
        self.vel_y = 0
        self.dead = True

In [81]:
def handle_input():
  keys = pygame.key.get_pressed()

  left = keys[pygame.K_a]
  right = keys[pygame.K_d]
  jump = keys[pygame.K_SPACE]

  return left, right, jump

In [82]:
SOLID_RECTS = (
    [pygame.Rect(x, y, PLATFORM_WIDTH, PLATFORM_HEIGHT) for x, y in PLATFORMS]
    + [pygame.Rect(x, y, GROUND_WIDTH, GROUND_HEIGHT) for x, y in GROUND]
    + [pygame.Rect(x, y, BELOW_WIDTH, BELOW_HEIGHT) for x, y in BELOW]
)

In [83]:
def handle_platform_collision(player, platform_rects):
    player.rect = player.get_rect()
    player.on_ground = False

    for platform_rect in platform_rects:
        if not player.rect.colliderect(platform_rect):
            continue

        overlap_top = player.rect.bottom - platform_rect.top
        overlap_bottom = platform_rect.bottom - player.rect.top
        overlap_left = player.rect.right - platform_rect.left
        overlap_right = platform_rect.right - player.rect.left

        if player.vel_y > 0 and overlap_top <= overlap_left and overlap_top <= overlap_right:
            player.y -= overlap_top
            player.vel_y = 0
            player.on_ground = True
        elif player.vel_y < 0 and overlap_bottom <= overlap_left and overlap_bottom <= overlap_right:
            player.y += overlap_bottom
            player.vel_y = 0
        elif overlap_left <= overlap_right:
            player.x -= overlap_left
        else:
            player.x += overlap_right

        player.rect = player.get_rect()

    return player.on_ground

In [84]:
class Enemy:

    def __init__(self, x, y,image, moveSpeed = 5):
        self.x = x
        self.y = y
        self.moveSpeed = moveSpeed
        self.image = image
        self.rect = pygame.Rect(self.x, self.y, image.get_width(), image.get_height())

    def move(self, minX, maxX):
        self.x += self.moveSpeed
        self.rect = pygame.Rect(self.x, self.y, self.image.get_width(), self.image.get_height())
        if self.x <=minX:
            self.image = pygame.transform.flip(self.image, True, False)
            self.moveSpeed = -self.moveSpeed

        if self.x >= maxX:
            self.image = pygame.transform.flip(self.image, True, False)
            self.moveSpeed = -self.moveSpeed

    def hitPlayer(self, player):
        player.rect = player.get_rect()
        
        if player.rect.colliderect(self.rect):
            player.dead = True


    

In [85]:
class CheckPoint:

    def __init__(self, x, y, image, level = 1):
        self.x = x
        self.y = y
        self.image = image
        self.level = level
        self.rect = pygame.Rect(self.x, self.y, image.get_width(), image.get_height())

    def win(self, player):
        player.rect = player.get_rect()
                
        if player.rect.colliderect(self.rect):
            return True

In [86]:
"""GROUND_TOLERANCE = 12

def is_Grounded(player: Player):
    player_left = player.x
    player_right = player.x + player.image.get_width()
    player_bottom = player.y + player.image.get_height()

    for x, y in PLATFORMS:
        landed_on_top = abs(player_bottom - y) <= GROUND_TOLERANCE
        within_platform_x = player_right > x and player_left < x + PLATFORM_WIDTH
        if landed_on_top and within_platform_x:
            return True

    for x, y in GROUND:
        landed_on_top = abs(player_bottom - y) <= GROUND_TOLERANCE
        within_ground_x = player_right > x and player_left < x + GROUND_WIDTH
        if landed_on_top and within_ground_x:
            return True

    return False
    """

'GROUND_TOLERANCE = 12\n\ndef is_Grounded(player: Player):\n    player_left = player.x\n    player_right = player.x + player.image.get_width()\n    player_bottom = player.y + player.image.get_height()\n\n    for x, y in PLATFORMS:\n        landed_on_top = abs(player_bottom - y) <= GROUND_TOLERANCE\n        within_platform_x = player_right > x and player_left < x + PLATFORM_WIDTH\n        if landed_on_top and within_platform_x:\n            return True\n\n    for x, y in GROUND:\n        landed_on_top = abs(player_bottom - y) <= GROUND_TOLERANCE\n        within_ground_x = player_right > x and player_left < x + GROUND_WIDTH\n        if landed_on_top and within_ground_x:\n            return True\n\n    return False\n    '

In [87]:
def skapa_nytt_konto():
    global användare

    with shelve.open("användare") as db:
        while användare in db:
            användare = input("Användarnamnet är redan taget. Skriv ett nytt användarnamn: ")

        db[användare] = hash(användare)
        print("Nu har din användare ", användare, "skapats")

    with shelve.open("karaktärsdata") as db:
        db[användare] = vald_karaktär
        print("Du börjar med denna karaktär: ", db[användare])


def logga_in():
    global användare

    with shelve.open("användare") as db:
        while användare not in db:
            print("Användarnamnet finns inte.")
            användare = input("Försök igen: ")

    with shelve.open("karaktärsdata") as db:
        karaktär = db[användare]

    print("Inloggad som", användare)
    print("Din tidigare valda karaktär är ", karaktär)

    return karaktär


server_id = input("Skriv IP-adress till önskad server: ")

val = input("Vill du skapa nytt konto eller logga in? 1 för att skapa konto och 2 för att logga in: ")

if val == "1":
    användare = input("Dags att skapa ett nytt konto, vad vill du heta? ")
    vald_karaktär = int(input("Välj karaktär (1-16): "))
    skapa_nytt_konto()
else:
    användare = input("Dags att logga in, vad är ditt användarnamn? ")
    vald_karaktär = logga_in()

Inloggad som Hallo
Din tidigare valda karaktär är  4


In [88]:
PLAYER_NAME = användare
SERVER_HOST = server_id
CHARACTER = vald_karaktär

try:
    net.close()
except NameError:
    pass

net = NetClient(SERVER_HOST, PLAYER_NAME, CHARACTER)
print("connected as player", net.my_id, "| char", net.char, "| resume", net.resume)


connected as player 1 | char 4 | resume (1040, 320.75)


In [89]:
enemy = Enemy(1000, 900,enemyImage)
enemy2 = Enemy(300, -1088,enemyImage)
enemy3 = Enemy(100, -3280,enemyImage, moveSpeed=7)
enemy4 = Enemy(100, -5950,enemyImage, moveSpeed=9)
enemy5 = Enemy(100, -6140,enemyImage, moveSpeed=9)
enemy6 = Enemy(100, -7220,enemyImage, moveSpeed=15)
enemy7 = Enemy(100, -7970,enemyImage, moveSpeed=15)
enemy8 = Enemy(100, -8360,enemyImage, moveSpeed=20)
enemy9 = Enemy(100, -8550,enemyImage, moveSpeed=20)
enemy10 = Enemy(100, -8930,enemyImage, moveSpeed=30)
enemy11 = Enemy(100, -9710,enemyImage, moveSpeed=30)


Enemies = [enemy, enemy2, enemy3, enemy4, enemy5, enemy6,
           enemy7, enemy8, enemy9, enemy10, enemy11]

In [90]:
camera = Camera()
player = Player(pick_character(net.char), camera)
checkPoint = CheckPoint(1600, -10100,exclamation_mark)
#Music
pygame.mixer_music.load("7pm.wav")
pygame.mixer_music.set_volume(1)
pygame.mixer_music.play(-1)


if net.resume is not None:
    player.x, player.y = net.resume

clock = pygame.time.Clock()

running = True
while running:
    for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
    
    left, right, jump = handle_input()
   
    player.move(left,right,jump)
    for ene in Enemies:
        ene.move(0, 2000)
        ene.hitPlayer(player)
    player.apply_gravity()
    player.on_ground = handle_platform_collision(player, SOLID_RECTS)

    
    remotes = net.update(player.x, player.y)
    if player.dead:
        player.x = 500
        player.y = 500
        camera.death_screen()
        pygame.display.flip()
        pygame.time.wait(3000)
        player.dead = False      
    else:
        camera.render_world(PLATFORMS, GROUND, BELOW, PUMPKINS, BUSHES, ORANGE_BUSHES, player, Enemies, checkPoint, remotes)

    status = "online" if net.connected else "OFFLINE"
    pygame.display.set_caption(
        f"{clock.get_fps():.0f} FPS | {PLAYER_NAME} #{net.my_id} | y = {round(player.y,0)}"
        f"{status} | {len(remotes)} others")
    pygame.display.flip()
    clock.tick(60)

net.close()
pygame.quit()